# E-Commerce Order Cancellation Risk Analysis

![E-Commerce Order Cancellation Risk Analysis](project_cover.png)

---


# E-Commerce Order Cancellation Risk Analysis

## 1. Business Understanding

### Business Problem
Order cancellations can create operational inefficiencies, increase costs, and negatively affect customer experience. Understanding the patterns associated with cancellations can help an e-commerce business investigate potential problems and prioritize operational reviews.

### Business Objective
Analyze historical e-commerce orders to identify characteristics associated with cancellations and develop a machine learning model that estimates the probability of cancellation for an eligible order.

### Business Questions
1. What percentage of orders are canceled?
2. Which product categories have higher cancellation rates?
3. How do payment methods relate to cancellation rates?
4. Are freight costs and estimated delivery windows associated with cancellations?
5. Which features are most informative for predicting cancellations?
6. Can orders be grouped into Low, Medium, and High cancellation-risk tiers?

### Project Objectives
- Understand the structure and quality of the Olist datasets.
- Perform exploratory data analysis.
- Prepare an order-level analytical dataset.
- Engineer a binary cancellation target.
- Train and evaluate an XGBoost classifier.
- Identify important predictive signals for operational investigation.
- Generate cancellation risk scores between 0% and 100%.
- Provide actionable, evidence-based business recommendations.

### Success Criteria
- Preserve one row per order in the analytical dataset.
- Prevent target leakage during model preparation.
- Evaluate the model using precision, recall, F1-score, PR-AUC, and a confusion matrix.
- Produce an order-level risk-scoring table and risk-tier summary.
- Clearly document findings, limitations, and recommendations.

### Project Scope
This project covers business understanding, data understanding, data preparation, modeling, and evaluation. Model deployment is not included.

### Important Limitation
Historical patterns and model feature importance can identify associations and predictive signals, but they do not independently prove the operational causes of cancellations.


## 2. Data Understanding

### 2.1 Dataset Access and Verification

The first step is to verify that all required datasets are available and accessible before beginning data exploration.

In [1]:
from pathlib import Path

# CRISP-DM Phase 2: Data Understanding
# Step 1: Verify the raw dataset files.

DATA_DIR = Path(
    r"C:\Users\admin\Downloads\Brazilian E-Commerce Public Dataset by Olist"
)

dataset_files = [
    "olist_orders_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "product_category_name_translation.csv",
]

print("Dataset directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.is_dir())

for filename in dataset_files:
    file_path = DATA_DIR / filename
    status = "FOUND" if file_path.is_file() else "MISSING"
    print(f"{status}: {filename}")

missing_files = [
    filename
    for filename in dataset_files
    if not (DATA_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Please check these missing files: {missing_files}"
    )

print("\nSuccess: all five required files are available.")

Dataset directory: C:\Users\admin\Downloads\Brazilian E-Commerce Public Dataset by Olist
Directory exists: True
FOUND: olist_orders_dataset.csv
FOUND: olist_order_payments_dataset.csv
FOUND: olist_order_items_dataset.csv
FOUND: olist_products_dataset.csv
FOUND: product_category_name_translation.csv

Success: all five required files are available.


### 2.2 Loading the Datasets

The Olist e-commerce datasets are loaded into Pandas DataFrames for initial inspection. This step helps us understand their structure, identify relevant variables, and prepare for data quality assessment.

In [2]:
import pandas as pd

# Load the five required datasets
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
category_translation = pd.read_csv(
    DATA_DIR / "product_category_name_translation.csv"
)

print("All five datasets loaded successfully.")

All five datasets loaded successfully.


### 2.3 Inspecting Dataset Dimensions

This step examines the number of rows and columns in each dataset to understand the volume and structure of the available data.

In [3]:
# Number of rows and columns in each dataset
datasets = {
    "Orders": orders,
    "Payments": payments,
    "Order Items": order_items,
    "Products": products,
    "Category Translation": category_translation,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

Orders: 99,441 rows × 8 columns
Payments: 103,886 rows × 5 columns
Order Items: 112,650 rows × 7 columns
Products: 32,951 rows × 9 columns
Category Translation: 71 rows × 2 columns


#### Dataset Dimensions Summary

The initial inspection identified five datasets with different sizes and structures:

| Dataset | Rows | Columns |
|---|---:|---:|
| Orders | 99,441 | 8 |
| Payments | 103,886 | 5 |
| Order Items | 112,650 | 7 |
| Products | 32,951 | 9 |
| Category Translation | 71 | 2 |

**Initial Observations**

- The orders dataset contains 99,441 order records and will serve as the primary dataset for this project.
- The payments dataset contains more rows than the orders dataset, suggesting that some orders may have multiple payment records.
- The order items dataset contains more rows than the orders dataset because an order can contain multiple products.
- The products dataset contains product-level information that can enrich the order analysis.
- The category translation dataset provides English translations for product category names.

These differences in row counts mean that the datasets must be aggregated or joined carefully to avoid duplicating order records during data preparation.

### 2.4 Previewing the Datasets

The first five records of each dataset are examined to understand the available variables, data formats, and structure before conducting further data quality checks.

In [4]:
# Preview the first five rows of each dataset
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name.upper())
    print("=" * 60)
    display(df.head())


ORDERS


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



PAYMENTS


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45



ORDER ITEMS


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14



PRODUCTS


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0



CATEGORY TRANSLATION


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


#### Summary of Dataset Preview

The initial preview confirmed that the five datasets contain complementary information required for the e-commerce order cancellation analysis. The orders dataset provides order statuses and timestamps, the payments dataset contains payment details, the order items dataset provides product prices and freight costs, and the products dataset contains product characteristics. The category translation dataset supports clearer interpretation by translating product categories into English.

The datasets have different levels of detail and row counts. Therefore, payment and order item records will need to be aggregated appropriately before being merged with the orders dataset to preserve one row per order.

These observations establish the foundation for the next step: inspecting data types, identifying missing values, and assessing data quality before data preparation.

### 2.5 Inspecting Data Types and Missing Values

This step examines the data types of each column and identifies missing values. The findings will help determine which variables require data type conversion, missing-value treatment, or further investigation before analysis.

In [5]:
# 2.5 Inspecting Data Types and Missing Values

for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name.upper())
    print("=" * 60)

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nTotal Missing Values:", df.isnull().sum().sum())



ORDERS

Data Types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Missing Values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Total Missing Values: 4908

PAYMENTS

Data Types:
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

Missing Values:
order_id                0
payment_sequential      0
payment_type            0
payment_install

#### Summary of Data Types and Missing Values

The initial data quality assessment identified missing values in the orders and products datasets, while the payments, order items, and category translation datasets contained no missing values.

The orders dataset contains missing approval, carrier handover, and customer delivery timestamps. These missing values may reflect different order outcomes and must be investigated before treatment. The products dataset contains missing category names, product descriptions, and physical measurements, which may affect category-based analysis and feature engineering.

All timestamp fields in the orders and order items datasets are currently stored as strings and will require appropriate datetime conversion during data preparation.

No missing values have been removed or imputed at this stage. Further checks for duplicate records and the meaning of missing values will guide subsequent cleaning decisions.

### 2.6 Checking for Duplicate Records

This step identifies duplicate rows and duplicate key values in the datasets. Duplicate checks are important for maintaining data integrity and preventing inaccurate counts or inflated totals during aggregation and merging.

In [6]:
# 2.6 Checking for Duplicate Records

for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name.upper())
    print("=" * 60)

    # Count fully duplicated rows
    full_duplicates = df.duplicated().sum()
    print("Fully duplicated rows:", full_duplicates)

    # Check duplicate values in the expected key column
    key_columns = {
        "Orders": "order_id",
        "Payments": "order_id",
        "Order Items": "order_id",
        "Products": "product_id",
        "Category Translation": "product_category_name",
    }

    key = key_columns[name]
    duplicate_keys = df[key].duplicated().sum()

    print(f"Rows with repeated {key} values:", duplicate_keys)



ORDERS
Fully duplicated rows: 0
Rows with repeated order_id values: 0

PAYMENTS
Fully duplicated rows: 0
Rows with repeated order_id values: 4446

ORDER ITEMS
Fully duplicated rows: 0
Rows with repeated order_id values: 13984

PRODUCTS
Fully duplicated rows: 0
Rows with repeated product_id values: 0

CATEGORY TRANSLATION
Fully duplicated rows: 0
Rows with repeated product_category_name values: 0


#### Summary of Duplicate Record Checks

The duplicate-checking process found no fully duplicated rows in any of the five datasets. The orders and products datasets contain unique primary identifiers, while the category translation dataset contains unique source category names.

Repeated order IDs were identified in the payments and order items datasets. These repetitions are expected because an order may have multiple payment records or contain multiple products. They should therefore not be removed simply because the order ID appears more than once.

During data preparation, payment and order-item records will be aggregated at the order level before being joined to the orders dataset. This will help preserve one row per order and prevent inflated counts or totals.

### 2.7 Summary of Data Understanding

The initial data understanding phase examined five datasets from the Olist Brazilian e-commerce platform: orders, payments, order items, products, and product category translations.

The orders dataset contains 99,441 records and will serve as the primary dataset for the analysis. Payments and order items contain multiple records per order, so they must be aggregated before merging to maintain an order-level analytical dataset.

The quality assessment identified missing timestamps in the orders dataset and missing product attributes in the products dataset. No fully duplicated rows were detected. Timestamp columns are currently stored as strings and will require appropriate conversion during data preparation.

The available data provides information on order status, purchase timing, payments, product categories, prices, and freight costs. These variables will support exploratory analysis of cancellation patterns.

The next phase will focus on data preparation, including appropriate aggregation, merging, feature engineering, missing-value handling, and the creation of a binary cancellation target. Features will be selected carefully to prevent target leakage and ensure that predictions rely on information available at the intended prediction time.